In [ ]:
#install llama-cpp-python on colab w gpu support, optional for gguf models
# !CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 50.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.8 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl size=35815396 sha256=d21be2c41376dbea288680f158f4941ec1ee39b043c2b34bdec4cda718cfe4b2
  Stored in directory: /root/.cache/pip/wheels/90/82/ab/8784ee3fb99ddb07fd36a679ddbe63122cc07718f6c1eb3be8
Successfully built llama-cpp-python


In [1]:
!pip install -q transformers peft bitsandbytes datasets tiktoken accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 30.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, PeftModel
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig
import pandas as pd
from datasets import Dataset


# Mount Google Drive
drive.mount('/content/drive')

# Verify the specific directory path
data_path = '/content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator/train'

if os.path.exists(data_path):
    print(f"Directory exists: {data_path}")
    print("Contents:", os.listdir(data_path))
else:
    print(f"Directory not found: {data_path}")

Mounted at /content/drive
Directory exists: /content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator/train
Contents: ['apache_nano_README.md', 'apache_nano_parsed_code.txt', 'apache_dubbo-rust_README.md', 'apache_dubbo-rust_parsed_code.txt', 'apache_rocketmq-client-go_README.md', 'apache_rocketmq-client-go_parsed_code.txt', 'apache_cordova-plugin-device_README.md', 'apache_cordova-plugin-device_parsed_code.txt', 'apache_maven-wrapper_README.md', 'apache_maven-wrapper_parsed_code.txt', 'apache_skywalking-eyes_README.md', 'apache_skywalking-eyes_parsed_code.txt', 'apache_predictionio-sdk-ruby_README.md', 'apache_predictionio-sdk-ruby_parsed_code.txt', 'apache_airflow-client-go_README.md', 'apache_airflow-client-go_parsed_code.txt', 'apache_calcite-avatica-go_README.md', 'apache_calcite-avatica-go_parsed_code.txt', 'apache_dubbo-python_README.md', 'apache_rocketmq-client-python_parsed_code.txt', 'apache_rocketmq-client-python_README.md', 'apache_dubbo-python_parsed_code.txt', 'apache_co

## Define Truncation Function

### Subtask:
Implement a helper function to truncate input text to a specific token limit.


**Reasoning**:
Implement a helper function `truncate_text` that uses a pre-trained tokenizer to ensure input text does not exceed a specified token limit.



In [3]:
# Initialize the tokenizer
# We use a public model checkpoint. If this fails due to gating, try "gpt2" as a fallback for simple token counting.
model_id = "Qwen/Qwen3-4B-Instruct-2507"

try:
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
except Exception as e:
    print(f"Failed to load {model_id}: {e}")
    print("Falling back to 'gpt2' tokenizer for truncation purposes.")
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

def truncate_text(text, max_tokens=80000):
    """
    Truncates the input text to a maximum number of tokens.

    Args:
        text (str): The input text to truncate.
        max_tokens (int): The maximum number of tokens allowed.

    Returns:
        str: The truncated text.
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)
    if len(tokens) > max_tokens:
        tokens = tokens[:max_tokens]
        text = tokenizer.decode(tokens)
    return text

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

## Prepare Dataset

### Subtask:
Scan the data directory for matching code and README pairs, apply truncation, format the data into a prompt structure, and create a Hugging Face Dataset.


**Reasoning**:
I will scan the directory for matching code and README files, truncate the code, format them into a prompt structure, and create a Hugging Face Dataset split into training and validation sets.



In [9]:
# Initialize an empty list to store the data samples
data_samples = []

# Iterate through the files in data_path

print(f"Scanning directory: {data_path}")

files = os.listdir(data_path)

for filename in files:
    if filename.endswith('_parsed_code.txt'):
        # Determine the corresponding README file name
        code_filename = filename
        readme_filename = filename.replace('_parsed_code.txt', '_README.md')

        code_file_path = os.path.join(data_path, code_filename)
        readme_file_path = os.path.join(data_path, readme_filename)

        # Check if the matching README file exists
        if os.path.exists(readme_file_path):
            try:
                # Read the content of the code file and the README file
                with open(code_file_path, 'r', encoding='utf-8') as f:
                    code_content = f.read()

                with open(readme_file_path, 'r', encoding='utf-8') as f:
                    readme_content = f.read()

                # Truncate the code content
                truncated_code = truncate_text(code_content)

                # Format the data into a prompt structure
                formatted_text = f"### Instruction:\nGenerate a README.md for the following code.\n\n### Input:\n{truncated_code}\n\n### Response:\n{readme_content}"

                # Append to the list
                data_samples.append({'text': formatted_text})
            except Exception as e:
                print(f"Error processing {filename}: {e}")

# Convert to DataFrame then to Hugging Face Dataset
if data_samples:
    df = pd.DataFrame(data_samples)
    dataset = Dataset.from_pandas(df)

    # Split the dataset into training and validation sets (90% train, 10% test)
    dataset = dataset.train_test_split(test_size=0.1)

    print("Dataset created successfully.")
    print(dataset)
    print("\nExample entry:")
    print(dataset['train'][0]['text'])
else:
    print("No matching code and README pairs found.")

Scanning directory: /content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator/train


Token indices sequence length is longer than the specified maximum sequence length for this model (1067167 > 1010000). Running this sequence through the model will result in indexing errors


Dataset created successfully.
DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 26
    })
    test: Dataset({
        features: ['text'],
        num_rows: 3
    })
})

Example entry:
### Instruction:
Generate a README.md for the following code.

### Input:
Tokens: 284742
Repo: apache/bahir-flink
Commit: 4a8f60f03f7caeb5421c7c88fa34a3ba419b61ee
Timestamp: 2026-03-04T19:51:45.901914

--- Repository Structure ---
.
├── .asf.yaml
├── .gitattributes
├── .github
│   └── workflows
│       └── maven-ci.yml
├── .gitignore
├── LICENSE
├── NOTICE
├── dev
│   ├── change-scala-version.sh
│   ├── checkstyle-suppressions.xml
│   ├── checkstyle.xml
│   └── release-build.sh
├── distribution
│   ├── pom.xml
│   └── src
│       └── main
│           └── assembly
│               └── src.xml
├── flink-connector-activemq
│   ├── pom.xml
│   └── src
│       ├── main
│       │   └── java
│       │       └── org
│       │           └── apache
│       │               └── flink
│  

## Setup Qwen Model & LoRA

### Subtask:
Load the Qwen/Qwen3-4B-Instruct-2507 model with 4-bit quantization and configure LoRA adapters.


In [11]:
# Define LoRA Config
# Qwen/Llama architectures benefit from targeting all linear layers
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

sft_config = SFTConfig(
    output_dir="/content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator/qwen-ft-adapter",
    max_length=None,        # This is where max input token length lives now
    dataset_text_field="text",  # Move this here too
    packing=False,
    num_train_epochs=3,
    per_device_train_batch_size=4,
)

trainer = SFTTrainer(
    model=model_id,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config)

# Train the model
print("Starting training for Qwen...")
trainer.train()

# Save the fine-tuned model (adapters)
adapter_path = "/content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator/qwen-ft-adapter"
trainer.save_model(adapter_path)
print(f"Qwen model adapters saved to {adapter_path}")

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/26 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/26 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/26 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training for Qwen...


Step,Training Loss
10,1.097377


Qwen model adapters saved to /content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator/qwen-ft-adapter


In [4]:
adapter_path = "/content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator/qwen-ft-adapter"
save_path = "./merged_model"

# 1. Load Base Model in FP16
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cpu" # Use CPU if you run out of VRAM; merging doesn't need a GPU
)

# 2. Load the Adapter
model = PeftModel.from_pretrained(base_model, adapter_path)

# 3. Merge and Unload (this combines the weights permanently)
merged_model = model.merge_and_unload()

# 4. Save the full model and tokenizer
merged_model.save_pretrained(save_path)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.save_pretrained(save_path)

print(f"Model merged and saved to {save_path}")

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model merged and saved to ./merged_model


In [5]:
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp
!pip install -r requirements.txt

Cloning into 'llama.cpp'...
remote: Enumerating objects: 81555, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 81555 (delta 17), reused 5 (delta 5), pack-reused 81522 (from 3)
Receiving objects: 100% (81555/81555), 308.37 MiB | 35.55 MiB/s, done.
Resolving deltas: 100% (58656/58656), done.
/content/llama.cpp
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━

In [6]:
#req.txt may have out of date toeknizers and will throw error if trying to run python script below so upgrading it
!pip install --upgrade transformers tokenizers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 32.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6


In [7]:
%cd llama.cpp
!python convert_hf_to_gguf.py ../merged_model --outtype f16 --outfile qwen3-4b-instruct-merged-f16.gguf

[Errno 2] No such file or directory: 'llama.cpp'
/content/llama.cpp
INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: Qwen3ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {2560, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {2560}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {9728, 2560}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> F16, shape = {2560, 9728}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> F16, shape = {2560, 9728}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {2560}
INFO:hf-to-gguf:blk.0.attn_k_norm.weight,  torch.float16 --> F32, shape = {128}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.float16 --> F16, shape = {25

In [ ]:
#download the gguf file
from google.colab import files
files.download('/content/llama.cpp/qwen3-4b-instruct-merged-f16.gguf')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
#move gguf model to google drive folder
!mv qwen3-4b-instruct-merged-f16.gguf /content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator/